## What this notebook does

Takes every patient (Geneva `PAT_XXXX` + Bern `ELXXX`), pulls their electrode coordinates out of FreeSurfer recon, and produces one combined "all electrodes on one brain" view plus per-patient mosaics.

It uses the cohort registry in `functions/lf_recon_shared_config.py` so EL and PAT are handled in the same loop. The default ignore list is `{"PAT_648", "PAT_699", "PAT_1145", "PAT_2856", "PAT_2893"}`. White-matter contacts are tagged via `C.load_wm_contacts(pid)` and saved as an `is_wm` flag, so every "combined" output exists in two flavours: with WM and without WM.

---

### Cell A — Per-patient `tkrRAS` export

For each kept patient:

- reads `elec_recon/<pid>.LEPTOVOX` and `elec_recon/<pid>.electrodeNames`
- applies the fixed LEPTOVOX convention (`PERM=(0,1,2)`, flip k only)
- converts voxel → `tkrRAS` using `<pid>/mri/brainmask.mgz`
- computes distance to pial + predicted hemisphere
- adds `is_wm` from the BIDS `*_electrodes.tsv`

**Outputs (per patient)** in
`OUT_ROOT/<pid>/glassbrain/`:

- `coords/<pid>_contacts_tkrRAS.csv` — coords + `is_wm` + provenance
- `png/<pid>_mosaic_LEPTOVOX.png` — 3-view native-brain mosaic

---

### Cell B — fsaverage surface group plot (the "all electrodes on one brain")

For each patient, maps native `tkrRAS` → nearest patient pial vertex → follows that vertex on `lh.sphere.reg` / `rh.sphere.reg` → nearest fsaverage sphere vertex → fsaverage pial xyz.

Uses one common fsaverage:
`\\...\#SHARE\To_send_collaborators\fsaverage`.

**Outputs** in `OUT_ROOT/fsaverage/`:

- `coords/<pid>_contacts_fsaverage.csv` (per patient)
- `coords/ALL_PATIENTS_contacts_fsaverage.csv` (combined, all contacts)
- `coords/ALL_PATIENTS_contacts_fsaverage_nowm.csv` (combined, WM dropped)
- `png/ALL_PATIENTS_fsaverage_with_wm.png`
- `png/ALL_PATIENTS_fsaverage_no_wm.png`

Each PNG is a 3-view (left / frontal / right) cortex with one colour per patient.

---

### Cell C — Talairach volumetric group plot

For each patient, applies their `mri/transforms/talairach.xfm` to the `tkrRAS` coords. Builds a template brain mesh once from the fsaverage `brainmask.mgz` (cached as `.vtk`) so all patients share one anatomical reference.

**Outputs** in `OUT_ROOT/talairach/`:

- `template/fsaverage_brainmask_surface.vtk` (cache)
- `coords/<pid>_contacts_talairach.csv` (per patient)
- `coords/ALL_PATIENTS_contacts_talairach.csv`
- `coords/ALL_PATIENTS_contacts_talairach_nowm.csv`
- `png/ALL_PATIENTS_talairach_with_wm.png`
- `png/ALL_PATIENTS_talairach_no_wm.png`

---

### Where everything lives

`OUT_ROOT = \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon`

## Recon contract for `outputs/250_recon` (my local folder sturcture)
```markdown

### 1. Per-patient native brain plots (PAT space)

**Patient root**

`\
asac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators\PAT_XXXX\`

**Required**

- Cortex surfaces (for plotting the brain)
  - `surf/lh.pial`
  - `surf/rh.pial`

- MRI for coordinate transforms
  - `mri/brainmask.mgz`

- Electrode positions (native reconstruction)
  - `elec_recon/PAT_XXXX.LEPTOVOX`
  - `elec_recon/PAT_XXXX.electrodeNames`

**Optional (native atlas / tissue labels)**

- `mri/aparc+aseg.mgz`  
- `mri/aparc.a2009s+aseg.mgz`  
- `mri/wmparc.mgz`  
- `mri/transforms/talairach.xfm`


### 2. Combined electrode coverage on fsaverage (group plot)

**Per-patient BIDS electrodes (already in fsaverage space)**

`\
asac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators\PAT_XXXX\BIDS\ieeg\`

- `sub-XXXX_electrodes.tsv`
- `sub-XXXX_coordsystem.json`

**fsaverage anatomy**

`\
asac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators\fsaverage\`

- `surf/lh.pial`
- `surf/rh.pial`

### 3. Parcellation-highlighted fsaverage plots (with or without electrodes)

**fsaverage surfaces + parcellations**

`\
asac-m2.unige.ch\m-HumanNeuronLab\#SHARE\To_send_collaborators\fsaverage\`

- Surfaces  
  - `surf/lh.pial`  
  - `surf/rh.pial`

- Parcellation annotation files  
  - `label/lh.aparc.annot`  
  - `label/rh.aparc.annot`  
  - `label/lh.aparc.a2009s.annot`  
  - `label/rh.aparc.a2009s.annot`  
  - `label/lh.aparc.DKTatlas.annot`  
  - `label/rh.aparc.DKTatlas.annot`

**Optional overlay**

- Same BIDS electrodes as in section 2:
  - `PAT_XXXX/BIDS/ieeg/sub-XXXX_electrodes.tsv`
```


## Part 1 — Per-patient tkrRAS export (Cell A)

Generates `<pid>_contacts_tkrRAS.csv` and a 3-view mosaic PNG for every patient (GVA + Bern), using the fixed LEPTOVOX convention (perm `(0,1,2)`, flip k only). WM contacts are flagged via `is_wm` from the BIDS `*_electrodes.tsv`.

In [2]:
# ============================================================
# CELL A — Per-patient tkrRAS export (cohort-aware, EL + PAT)
#
# For every patient in C.PATIENTS (minus PATIENTS_IGNORED),
# read LEPTOVOX + electrodeNames, apply the fixed LEPTOVOX convention
# (perm=(0,1,2), flip k only), convert voxel -> tkrRAS, and write:
#   OUT_ROOT/<pid>/glassbrain/coords/<pid>_contacts_tkrRAS.csv
#   OUT_ROOT/<pid>/glassbrain/png/<pid>_mosaic_LEPTOVOX.png
#
# WM membership (from BIDS *_electrodes.tsv) is added as an "is_wm"
# column so downstream cells can toggle without re-deriving.
# ============================================================

from pathlib import Path
import importlib

import numpy as np
import pandas as pd
import nibabel as nib
import pyvista as pv
import imageio.v3 as iio
from nibabel.freesurfer.io import read_geometry
from scipy.spatial import cKDTree

from functions import lf_recon_shared_config as C
importlib.reload(C)

# -------------------------
# CONFIG
# -------------------------
OUT_ROOT = Path(
    r"\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM"
    r"\Analysis_LoraFanda\02_FBM_Clustering\outputs\250_recon"
)

# Default ignore list (kept here so it is the single source of truth for the
# whole notebook; later cells just import this).
PATIENTS_IGNORED = {"PAT_648", "PAT_699", "PAT_1145", "PAT_2856", "PAT_2893"}

# Fixed LEPTOVOX convention
PERM  = (0, 1, 2)
FLIPS = (False, False, True)  # flip k only

VIEWS          = ("left", "frontal", "right")
WINDOW_SIZE    = (1200, 1000)
TRANSPARENT_BG = True
BRAIN_COLOR    = "#ead6db"
BRAIN_OPACITY  = 0.25
POINT_COLOR    = "purple"
POINT_SIZE     = 10
POINT_OPACITY  = 0.9

OVERWRITE_CSV = True
OVERWRITE_PNG = True


# -------------------------
# IO + geometry helpers
# -------------------------
def read_electrode_lines_drop2(path: Path) -> list[str]:
    lines = [ln.strip() for ln in path.read_text(encoding="utf-8", errors="ignore").splitlines() if ln.strip()]
    if len(lines) < 3:
        raise ValueError(f"electrodeNames too short: {path}")
    return lines[2:]  # drop timestamp + header

def parse_name_and_hemi(lines):
    names_raw, names_clean, hemi = [], [], []
    for ln in lines:
        parts = ln.split()
        nm = parts[0] if parts else ln
        h = None
        if parts:
            last = parts[-1].upper()
            if last in ("L", "R"):
                h = last
        names_raw.append(ln); names_clean.append(nm); hemi.append(h)
    return (np.array(names_raw, dtype=object),
            np.array(names_clean, dtype=object),
            np.array(hemi, dtype=object))

def read_leptovox_xyz(path: Path) -> np.ndarray:
    rows = []
    for ln in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        t = ln.strip()
        if not t or t.startswith("#"):
            continue
        parts = t.replace(",", " ").split()
        if len(parts) < 3:
            continue
        try:
            rows.append((float(parts[0]), float(parts[1]), float(parts[2])))
        except ValueError:
            continue
    if not rows:
        raise ValueError(f"No numeric rows found in {path}")
    return np.asarray(rows, dtype=float)

def pick_mgz(subj_dir: Path) -> Path:
    for cand in ["brainmask.mgz", "T1.mgz", "orig.mgz"]:
        p = subj_dir / "mri" / cand
        if p.is_file():
            return p
    raise FileNotFoundError(f"Missing brainmask/T1/orig in {subj_dir/'mri'}")

def voxel_to_tkr(points_ijk, vox2ras_tkr):
    n = points_ijk.shape[0]
    ijk_h = np.c_[points_ijk, np.ones(n)]
    return (vox2ras_tkr @ ijk_h.T).T[:, :3]

def apply_voxel_flips(ijk, vol_shape, flips):
    dims = np.array(vol_shape, dtype=float)
    out = ijk.copy()
    for ax, do_flip in enumerate(flips):
        if do_flip:
            out[:, ax] = (dims[ax] - 1.0) - out[:, ax]
    return out

def nearest_pial_metrics(points_tkr, lh_v, rh_v):
    kdl = cKDTree(lh_v); kdr = cKDTree(rh_v)
    dl, _ = kdl.query(points_tkr, k=1, workers=-1)
    dr, _ = kdr.query(points_tkr, k=1, workers=-1)
    is_left = dl <= dr
    return np.minimum(dl, dr), is_left


# -------------------------
# Render helpers
# -------------------------
def make_mesh(v, f):
    return pv.PolyData(v, np.c_[np.full(len(f), 3), f].astype(np.int64))

def set_view(pl, view):
    if view == "left":     pl.view_yz(negative=True)
    elif view == "frontal":pl.view_xz(negative=False)
    elif view == "right":  pl.view_yz(negative=False)
    else: raise ValueError(view)
    pl.camera.zoom(1.15)

def render_view(lh_mesh, rh_mesh, points_tkr, view):
    pl = pv.Plotter(off_screen=True, window_size=WINDOW_SIZE)
    pl.set_background("white")
    pl.add_mesh(lh_mesh, color=BRAIN_COLOR, opacity=BRAIN_OPACITY, smooth_shading=True)
    pl.add_mesh(rh_mesh, color=BRAIN_COLOR, opacity=BRAIN_OPACITY, smooth_shading=True)
    pl.add_points(points_tkr, color=POINT_COLOR, render_points_as_spheres=True,
                  point_size=POINT_SIZE, opacity=POINT_OPACITY)
    set_view(pl, view)
    img = pl.screenshot(transparent_background=TRANSPARENT_BG, return_img=True)
    pl.close()
    return img

def stitch_horiz(imgs):
    H = min(im.shape[0] for im in imgs)
    return np.concatenate([im[:H] for im in imgs], axis=1)


# -------------------------
# Per-patient export
# -------------------------
def export_and_mosaic_patient(pid: str) -> dict:
    subj_dir = C.patient_recon_dir(pid)

    # Geneva uses "PAT_XXXX.LEPTOVOX"; Bern uses "ELXXX.LEPTOVOX" — both follow {pid}.*
    names_path = subj_dir / "elec_recon" / f"{pid}.electrodeNames"
    vox_path   = subj_dir / "elec_recon" / f"{pid}.LEPTOVOX"
    if not names_path.is_file(): raise FileNotFoundError(f"{pid}: missing {names_path}")
    if not vox_path.is_file():   raise FileNotFoundError(f"{pid}: missing {vox_path}")

    mgz = pick_mgz(subj_dir)
    img = nib.load(str(mgz))
    vox2ras_tkr = img.header.get_vox2ras_tkr()
    vol_shape   = img.shape[:3]

    lh_v, lh_f = read_geometry(str(subj_dir / "surf" / "lh.pial"))
    rh_v, rh_f = read_geometry(str(subj_dir / "surf" / "rh.pial"))
    lh_mesh, rh_mesh = make_mesh(lh_v, lh_f), make_mesh(rh_v, rh_f)

    lines = read_electrode_lines_drop2(names_path)
    name_raw, name_clean, hemi_expected = parse_name_and_hemi(lines)

    pts = read_leptovox_xyz(vox_path)
    if pts.shape[0] != len(name_raw):
        raise RuntimeError(f"{pid}: LEPTOVOX rows ({pts.shape[0]}) != electrodeNames contacts ({len(name_raw)})")

    # 0/1-based detection
    mins = pts.min(axis=0); maxs = pts.max(axis=0)
    dims = np.array(vol_shape, dtype=float)
    looks_one_based  = (mins >= 1).all() and (maxs <= dims).all()
    looks_zero_based = (mins >= 0).all() and (maxs <  dims).all()
    pts_ijk = pts.copy()
    index_mode = "0-based (assumed)"
    if looks_one_based and not looks_zero_based:
        pts_ijk -= 1.0
        index_mode = "1-based->0-based"

    pts_ijk = pts_ijk[:, PERM]
    pts_ijk = apply_voxel_flips(pts_ijk, vol_shape, FLIPS)
    pts_tkr = voxel_to_tkr(pts_ijk, vox2ras_tkr)
    dist_mm, pred_is_left = nearest_pial_metrics(pts_tkr, lh_v, rh_v)

    # WM flag from BIDS *_electrodes.tsv (strict 100% WM)
    try:
        wm_names = C.load_wm_contacts(pid)
    except Exception as e:
        print(f"  [warn] {pid}: load_wm_contacts failed -> all is_wm=False ({e})")
        wm_names = set()
    is_wm = np.array([nm in wm_names for nm in name_clean], dtype=bool)

    out_coords = OUT_ROOT / pid / "glassbrain" / "coords"
    out_pngdir = OUT_ROOT / pid / "glassbrain" / "png"
    out_coords.mkdir(parents=True, exist_ok=True)
    out_pngdir.mkdir(parents=True, exist_ok=True)
    out_csv = out_coords / f"{pid}_contacts_tkrRAS.csv"
    out_png = out_pngdir / f"{pid}_mosaic_LEPTOVOX.png"

    if OVERWRITE_CSV or not out_csv.is_file():
        df_out = pd.DataFrame({
            "patient": pid,
            "cohort": C.cohort_of(pid),
            "name_raw": name_raw,
            "name": name_clean,
            "hemi_expected": hemi_expected,
            "x": pts_tkr[:,0], "y": pts_tkr[:,1], "z": pts_tkr[:,2],
            "pred_isLeft": pred_is_left.astype(int),
            "dist_to_pial_mm": np.round(dist_mm, 2),
            "is_wm": is_wm.astype(int),
            "source_space": "tkrRAS",
            "source_provenance": (
                f"LEPTOVOX; {index_mode}; perm={PERM}; "
                f"flips={tuple(int(b) for b in FLIPS)} (flip k only)"
            ),
            "leptovox_file": str(vox_path),
            "mgz_used": str(mgz),
        })
        df_out.to_csv(out_csv, index=False)

    if OVERWRITE_PNG or not out_png.is_file():
        imgs = [render_view(lh_mesh, rh_mesh, pts_tkr, v) for v in VIEWS]
        iio.imwrite(out_png, stitch_horiz(imgs))

    return {
        "pid": pid, "cohort": C.cohort_of(pid), "status": "OK",
        "n_contacts": int(len(name_raw)),
        "n_wm": int(is_wm.sum()),
        "median_dist_to_pial_mm": float(np.median(dist_mm)),
        "pct_pred_left": float(np.mean(pred_is_left) * 100.0),
        "index_mode": index_mode,
        "csv": str(out_csv), "png": str(out_png),
    }


# -------------------------
# Run for ALL patients (EL + PAT) using the cohort registry
# -------------------------
def _discover_patients():
    """Discover all patient IDs from the FreeSurfer cohort roots (EL + PAT)."""
    pats = []
    try:
        if C.ROOT_PAT.exists():
            pats += [p.name for p in C.ROOT_PAT.iterdir() if p.is_dir() and p.name.startswith("PAT_")]
    except Exception:
        pass
    try:
        if C.ROOT_BERN_FS.exists():
            pats += [p.name for p in C.ROOT_BERN_FS.iterdir() if p.is_dir() and p.name.startswith("EL")]
    except Exception:
        pass
    return sorted(set(pats))

patient_ids = [pid for pid in _discover_patients() if pid not in PATIENTS_IGNORED]
print(f"[INFO] Running on {len(patient_ids)} patients ({sum(1 for p in patient_ids if p.startswith('PAT_'))} GVA + {sum(1 for p in patient_ids if p.startswith('EL'))} BERN)")

rows = []
for pid in patient_ids:
    try:
        rows.append(export_and_mosaic_patient(pid))
        print(f"  [OK]    {pid}: {rows[-1]['n_contacts']} contacts ({rows[-1]['n_wm']} WM)")
    except Exception as e:
        rows.append({"pid": pid, "status": "ERROR", "error": str(e)})
        print(f"  [ERROR] {pid}: {e}")

df_status = pd.DataFrame(rows)
df_status


[INFO] Running on 18 patients (18 GVA + 0 BERN)
  [ERROR] PAT_2868: module 'functions.lf_recon_shared_config' has no attribute 'patient_recon_dir'
  [ERROR] PAT_3066: module 'functions.lf_recon_shared_config' has no attribute 'patient_recon_dir'
  [ERROR] PAT_3301: module 'functions.lf_recon_shared_config' has no attribute 'patient_recon_dir'
  [ERROR] PAT_3390: module 'functions.lf_recon_shared_config' has no attribute 'patient_recon_dir'
  [ERROR] PAT_3415: module 'functions.lf_recon_shared_config' has no attribute 'patient_recon_dir'
  [ERROR] PAT_3455: module 'functions.lf_recon_shared_config' has no attribute 'patient_recon_dir'
  [ERROR] PAT_3780: module 'functions.lf_recon_shared_config' has no attribute 'patient_recon_dir'
  [ERROR] PAT_3780_good: module 'functions.lf_recon_shared_config' has no attribute 'patient_recon_dir'
  [ERROR] PAT_3965: module 'functions.lf_recon_shared_config' has no attribute 'patient_recon_dir'
  [ERROR] PAT_3975: module 'functions.lf_recon_shared_co

,pid,status,error
0,PAT_2868,ERROR,module 'functions.lf_recon_shared_config' has ...
1,PAT_3066,ERROR,module 'functions.lf_recon_shared_config' has ...
2,PAT_3301,ERROR,module 'functions.lf_recon_shared_config' has ...
3,PAT_3390,ERROR,module 'functions.lf_recon_shared_config' has ...
4,PAT_3415,ERROR,module 'functions.lf_recon_shared_config' has ...
5,PAT_3455,ERROR,module 'functions.lf_recon_shared_config' has ...
6,PAT_3780,ERROR,module 'functions.lf_recon_shared_config' has ...
7,PAT_3780_good,ERROR,module 'functions.lf_recon_shared_config' has ...
8,PAT_3965,ERROR,module 'functions.lf_recon_shared_config' has ...
9,PAT_3975,ERROR,module 'functions.lf_recon_shared_config' has ...


## Part 2 — fsaverage surface group mapping (Cell B)

Maps every patient's tkrRAS contacts onto a single fsaverage cortex (the `#SHARE\\To_send_collaborators\\fsaverage` one) via patient pial → `sphere.reg` → fsaverage `sphere.reg` → fsaverage pial. Produces a combined CSV and a group PNG, both in `with_wm` and `no_wm` flavors.

In [3]:
# ============================================================
# CELL B — fsaverage surface group mapping (EL + PAT combined)
#
# For each kept patient: tkrRAS -> nearest patient pial vertex ->
# follow that vertex on lh/rh.sphere.reg -> nearest fsaverage sphere
# vertex -> fsaverage pial xyz.
#
# Single fsaverage target = #SHARE\To_send_collaborators\fsaverage
# (per user choice, used for BOTH GVA and BERN cohorts).
#
# Outputs two PNGs and two CSVs:
#   * with WM contacts        (suffix "with_wm")
#   * without WM contacts     (suffix "no_wm")
# ============================================================

from pathlib import Path
import importlib

import numpy as np
import pandas as pd
import pyvista as pv
import imageio.v3 as iio
from nibabel.freesurfer.io import read_geometry
from scipy.spatial import cKDTree

from functions import lf_recon_shared_config as C
importlib.reload(C)

# Reuse OUT_ROOT, PATIENTS_IGNORED, render helpers from Cell A
# (they are in the notebook namespace once Cell A has been run)

# Single, canonical fsaverage for the group plot
FSAVERAGE = C.SHARED_RECON_ROOT_GVA / "fsaverage"
print(f"[INFO] Using fsaverage: {FSAVERAGE}")

# Patient-coloring palette
PATIENT_COLORS = [
    "blueviolet","fuchsia","deeppink","crimson","pink","red","chocolate","gold",
    "purple","saddlebrown","lemonchiffon","lavenderblush","lime","powderblue",
    "forestgreen","lightcyan","navy","darkslategray","black","darkred",
    "darkolivegreen","aquamarine","teal","orange","cyan","magenta","yellow",
]


# -------------------------
# fsaverage surfaces (load once)
# -------------------------
fs_lh_v, fs_lh_f = read_geometry(str(FSAVERAGE / "surf" / "lh.pial"))
fs_rh_v, fs_rh_f = read_geometry(str(FSAVERAGE / "surf" / "rh.pial"))
fs_lh_sph, _     = read_geometry(str(FSAVERAGE / "surf" / "lh.sphere.reg"))
fs_rh_sph, _     = read_geometry(str(FSAVERAGE / "surf" / "rh.sphere.reg"))
fs_lh_kd = cKDTree(fs_lh_sph)
fs_rh_kd = cKDTree(fs_rh_sph)

fs_lh_mesh = make_mesh(fs_lh_v, fs_lh_f)
fs_rh_mesh = make_mesh(fs_rh_v, fs_rh_f)


# 252-style cohort-aware patient palette (uses C.EL_COLOR_NAMES / C.PAT_COLOR_NAMES if available;
# falls back to PATIENT_COLORS for unknown cohorts). Ensures plots in this notebook share
# the same per-patient colors as the 252_clustering_recon notebook.
import matplotlib.colors as _mcolors
import matplotlib.pyplot as _plt

def _safe_to_rgba(name, fallback=(0.5, 0.5, 0.5, 1.0)):
    try:
        return _mcolors.to_rgba(name)
    except (ValueError, KeyError):
        return fallback

def _palette_patients(patient_ids):
    """Cohort-aware patient palette matching 252_clustering_recon."""
    el_names  = list(getattr(C, 'EL_COLOR_NAMES', []))
    pat_names = list(getattr(C, 'PAT_COLOR_NAMES', []))
    el_pats   = sorted({p for p in patient_ids if str(p).upper().startswith('EL')})
    pat_pats  = sorted({p for p in patient_ids if str(p).upper().startswith('PAT')})
    other     = sorted({p for p in patient_ids if p not in el_pats and p not in pat_pats})
    palette   = {}
    for i, p in enumerate(el_pats):
        nm = el_names[i % len(el_names)] if el_names else 'tab:blue'
        palette[p] = _safe_to_rgba(nm)
    for i, p in enumerate(pat_pats):
        nm = pat_names[i % len(pat_names)] if pat_names else 'tab:red'
        palette[p] = _safe_to_rgba(nm)
    for i, p in enumerate(other):
        cmap = _plt.get_cmap('tab10')
        palette[p] = cmap(i % 10)
    uniq = el_pats + pat_pats + other
    return palette, uniq

def render_fsaverage_by_patient(df, out_png, *, point_size=10, opacity=0.9, brain_opacity=0.2, brain_color="#cdc8b1"):
    imgs = []
    patients = sorted(df["patient"].dropna().unique())
    _palette, _uniq_pats = _palette_patients(patients)
    for view in VIEWS:
        pl = pv.Plotter(off_screen=True, window_size=WINDOW_SIZE)
        pl.set_background("white")
        pl.add_mesh(fs_lh_mesh, color=brain_color, opacity=brain_opacity, smooth_shading=True)
        pl.add_mesh(fs_rh_mesh, color=brain_color, opacity=brain_opacity, smooth_shading=True)
        for i, pid in enumerate(patients):
            dfp = df[df["patient"] == pid]
            if not len(dfp):
                continue
            pl.add_points(
                dfp[["x","y","z"]].to_numpy(float),
                color=_palette.get(pid, (0.5, 0.5, 0.5, 1.0)),
                render_points_as_spheres=True,
                point_size=point_size, opacity=opacity,
                label=pid,
            )
        pl.reset_camera()
        set_view(pl, view)
        img = pl.screenshot(transparent_background=TRANSPARENT_BG, return_img=True)
        pl.close()
        imgs.append(img)
    iio.imwrite(out_png, stitch_horiz(imgs))
    return out_png


# -------------------------
# Map every patient's tkrRAS contacts to fsaverage
# -------------------------
patient_ids = sorted([pid for pid in C.PATIENTS if pid not in PATIENTS_IGNORED])
print(f"[INFO] Mapping {len(patient_ids)} patients to fsaverage...")

fs_rows = []
for pid in patient_ids:
    subj_dir = C.patient_recon_dir(pid)
    csv_in = OUT_ROOT / pid / "glassbrain" / "coords" / f"{pid}_contacts_tkrRAS.csv"
    if not csv_in.is_file():
        print(f"  [SKIP] {pid}: missing {csv_in.name} (run Cell A first)")
        continue

    # Patient surfaces + spheres
    lh_v, _   = read_geometry(str(subj_dir / "surf" / "lh.pial"))
    rh_v, _   = read_geometry(str(subj_dir / "surf" / "rh.pial"))
    lh_sph, _ = read_geometry(str(subj_dir / "surf" / "lh.sphere.reg"))
    rh_sph, _ = read_geometry(str(subj_dir / "surf" / "rh.sphere.reg"))
    lh_kd = cKDTree(lh_v); rh_kd = cKDTree(rh_v)

    df = pd.read_csv(csv_in)
    pts = df[["x","y","z"]].to_numpy(float)

    rows_out = []
    for i, p in enumerate(pts):
        dl, il = lh_kd.query(p)
        dr, ir = rh_kd.query(p)
        if dl <= dr:
            hemi = "L"; sph = lh_sph[il]
            _, fs_vtx = fs_lh_kd.query(sph)
            fs_xyz = fs_lh_v[fs_vtx]
        else:
            hemi = "R"; sph = rh_sph[ir]
            _, fs_vtx = fs_rh_kd.query(sph)
            fs_xyz = fs_rh_v[fs_vtx]

        rows_out.append({
            "patient": pid,
            "cohort": C.cohort_of(pid),
            "name": df.iloc[i]["name"],
            "name_raw": df.iloc[i].get("name_raw", df.iloc[i]["name"]),
            "hemi": hemi,
            "x": float(fs_xyz[0]), "y": float(fs_xyz[1]), "z": float(fs_xyz[2]),
            "is_wm": int(df.iloc[i].get("is_wm", 0)),
        })

    df_fs = pd.DataFrame(rows_out)
    out_dir = OUT_ROOT / "fsaverage" / "coords"
    out_dir.mkdir(parents=True, exist_ok=True)
    df_fs.to_csv(out_dir / f"{pid}_contacts_fsaverage.csv", index=False)
    fs_rows.append(df_fs)
    print(f"  [OK] {pid}: {len(df_fs)} contacts ({df_fs['is_wm'].sum()} WM)")

if not fs_rows:
    raise RuntimeError("No fsaverage rows produced. Run Cell A first.")

df_all_fs = pd.concat(fs_rows, ignore_index=True)
out_all_dir = OUT_ROOT / "fsaverage" / "coords"
out_all_dir.mkdir(parents=True, exist_ok=True)
df_all_fs.to_csv(out_all_dir / "ALL_PATIENTS_contacts_fsaverage.csv", index=False)

# WM toggle: produce two CSVs and two group PNGs
df_no_wm = df_all_fs[df_all_fs["is_wm"] == 0].copy()
df_no_wm.to_csv(out_all_dir / "ALL_PATIENTS_contacts_fsaverage_nowm.csv", index=False)

png_dir = OUT_ROOT / "fsaverage" / "png"
png_dir.mkdir(parents=True, exist_ok=True)
render_fsaverage_by_patient(df_all_fs, png_dir / "ALL_PATIENTS_fsaverage_with_wm.png")
render_fsaverage_by_patient(df_no_wm,  png_dir / "ALL_PATIENTS_fsaverage_no_wm.png")

print(f"\n[DONE] fsaverage group plots:")
print(f"  with WM:    {png_dir/'ALL_PATIENTS_fsaverage_with_wm.png'}  (n={len(df_all_fs)})")
print(f"  without WM: {png_dir/'ALL_PATIENTS_fsaverage_no_wm.png'}    (n={len(df_no_wm)})")
df_all_fs.head()


AttributeError: module 'functions.lf_recon_shared_config' has no attribute 'SHARED_RECON_ROOT_GVA'

## Part 3 — Talairach volumetric group mapping (Cell C)

Applies each patient's `talairach.xfm` to their tkrRAS contacts and renders them on a template brain derived from the fsaverage `brainmask.mgz`. Produces a combined CSV and a group PNG, both in `with_wm` and `no_wm` flavors.

In [4]:
# ============================================================
# CELL C — Talairach volumetric group mapping (EL + PAT combined)
#
# For each kept patient: tkrRAS -> apply talairach.xfm -> Tal/MNI-ish
# space. Render points on a single template brain surface derived
# from the fsaverage brainmask.mgz (for an anatomical reference).
#
# WM toggle produces two CSVs and two group PNGs (with / without WM).
# ============================================================

from pathlib import Path
import importlib

import numpy as np
import pandas as pd
import nibabel as nib
import pyvista as pv
import imageio.v3 as iio

from functions import lf_recon_shared_config as C
importlib.reload(C)

# Reuse OUT_ROOT, PATIENTS_IGNORED, FSAVERAGE, render helpers from Cells A/B

# -------------------------
# Helpers: talairach.xfm parser
# -------------------------
def load_talairach_xfm(xfm_path: Path) -> np.ndarray:
    lines = xfm_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    start = None
    for i, ln in enumerate(lines):
        if ln.strip().startswith("Linear_Transform"):
            start = i + 1
            break
    if start is None:
        raise RuntimeError(f"Could not find Linear_Transform in {xfm_path}")
    rows = []
    for j in range(3):
        raw = lines[start + j].strip()
        parts = [p.rstrip(";") for p in raw.split()]
        if len(parts) != 4:
            raise RuntimeError(f"Invalid transform row in {xfm_path}: {raw}")
        rows.append([float(p) for p in parts])
    M = np.eye(4)
    M[:3, :4] = np.array(rows, dtype=float)
    return M

def apply_affine(points_xyz: np.ndarray, M: np.ndarray) -> np.ndarray:
    n = points_xyz.shape[0]
    return (M @ np.c_[points_xyz, np.ones(n)].T).T[:, :3]


# -------------------------
# Build a template brain surface from fsaverage brainmask.mgz (cached)
# -------------------------
def build_template_brain_surface(out_vtk: Path) -> pv.PolyData:
    if out_vtk.is_file():
        return pv.read(str(out_vtk))
    brainmask = FSAVERAGE / "mri" / "brainmask.mgz"
    if not brainmask.is_file():
        raise FileNotFoundError(f"Missing template volume: {brainmask}")

    img = nib.load(str(brainmask))
    vol = np.asanyarray(img.dataobj)
    vox2ras = img.affine

    grid = pv.ImageData(dimensions=np.array(vol.shape) + 1)
    grid.spacing = (1.0, 1.0, 1.0)
    grid.origin  = (0.0, 0.0, 0.0)
    grid.cell_data["mask"] = vol.flatten(order="F")
    grid = grid.cell_data_to_point_data()
    surf = grid.contour(isosurfaces=[0.5], scalars="mask")

    pts = surf.points
    ras = (vox2ras @ np.c_[pts, np.ones(len(pts))].T).T[:, :3]
    surf.points = ras

    out_vtk.parent.mkdir(parents=True, exist_ok=True)
    surf.save(str(out_vtk))
    return surf


# -------------------------
# Render
# -------------------------
def render_talairach_by_patient(df, brain_mesh, out_png, *, point_size=10, opacity=0.9, brain_opacity=0.18):
    imgs = []
    patients = sorted(df["patient"].dropna().unique())
    for view in VIEWS:
        pl = pv.Plotter(off_screen=True, window_size=WINDOW_SIZE)
        pl.set_background("white")
        pl.add_mesh(brain_mesh, color=BRAIN_COLOR, opacity=brain_opacity)
        for i, pid in enumerate(patients):
            dfp = df[df["patient"] == pid]
            if not len(dfp): continue
            pl.add_points(
                dfp[["x","y","z"]].to_numpy(float),
                color=PATIENT_COLORS[i % len(PATIENT_COLORS)],
                render_points_as_spheres=True,
                point_size=point_size, opacity=opacity,
                label=pid,
            )
        pl.reset_camera()
        set_view(pl, view)
        img = pl.screenshot(transparent_background=TRANSPARENT_BG, return_img=True)
        pl.close()
        imgs.append(img)
    iio.imwrite(out_png, stitch_horiz(imgs))
    return out_png


# -------------------------
# Run
# -------------------------
tmpl_dir  = OUT_ROOT / "talairach" / "template"
brain_mesh = build_template_brain_surface(tmpl_dir / "fsaverage_brainmask_surface.vtk")

patient_ids = sorted([pid for pid in C.PATIENTS if pid not in PATIENTS_IGNORED])
print(f"[INFO] Mapping {len(patient_ids)} patients to Talairach...")

tal_rows = []
for pid in patient_ids:
    subj_dir = C.patient_recon_dir(pid)
    csv_in = OUT_ROOT / pid / "glassbrain" / "coords" / f"{pid}_contacts_tkrRAS.csv"
    xfm    = subj_dir / "mri" / "transforms" / "talairach.xfm"
    if not csv_in.is_file():
        print(f"  [SKIP] {pid}: missing {csv_in.name} (run Cell A first)"); continue
    if not xfm.is_file():
        print(f"  [SKIP] {pid}: missing {xfm}"); continue

    df = pd.read_csv(csv_in)
    pts_tkr = df[["x","y","z"]].to_numpy(float)
    M = load_talairach_xfm(xfm)
    pts_tal = apply_affine(pts_tkr, M)

    df_out = df.copy()
    df_out[["x","y","z"]] = pts_tal
    df_out["space"]     = "Talairach"
    df_out["transform"] = "FreeSurfer talairach.xfm"
    if "patient" not in df_out.columns:
        df_out["patient"] = pid
    if "cohort" not in df_out.columns:
        df_out["cohort"] = C.cohort_of(pid)

    out_dir = OUT_ROOT / "talairach" / "coords"
    out_dir.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(out_dir / f"{pid}_contacts_talairach.csv", index=False)
    tal_rows.append(df_out)
    print(f"  [OK] {pid}: {len(df_out)} contacts ({int(df_out.get('is_wm', pd.Series([0])).sum())} WM)")

if not tal_rows:
    raise RuntimeError("No Talairach rows produced. Run Cell A first.")

df_all_tal = pd.concat(tal_rows, ignore_index=True)
out_all_dir = OUT_ROOT / "talairach" / "coords"
out_all_dir.mkdir(parents=True, exist_ok=True)
df_all_tal.to_csv(out_all_dir / "ALL_PATIENTS_contacts_talairach.csv", index=False)

df_no_wm = df_all_tal[df_all_tal["is_wm"] == 0].copy() if "is_wm" in df_all_tal.columns else df_all_tal.copy()
df_no_wm.to_csv(out_all_dir / "ALL_PATIENTS_contacts_talairach_nowm.csv", index=False)

png_dir = OUT_ROOT / "talairach" / "png"
png_dir.mkdir(parents=True, exist_ok=True)
render_talairach_by_patient(df_all_tal, brain_mesh, png_dir / "ALL_PATIENTS_talairach_with_wm.png")
render_talairach_by_patient(df_no_wm,   brain_mesh, png_dir / "ALL_PATIENTS_talairach_no_wm.png")

print(f"\n[DONE] Talairach group plots:")
print(f"  with WM:    {png_dir/'ALL_PATIENTS_talairach_with_wm.png'}  (n={len(df_all_tal)})")
print(f"  without WM: {png_dir/'ALL_PATIENTS_talairach_no_wm.png'}    (n={len(df_no_wm)})")
df_all_tal.head()


NameError: name 'FSAVERAGE' is not defined